In [0]:
# Verificar que la tabla está accesible
df_bronze = spark.table("workspace.default.london_price_paid")

print(f"Filas: {df_bronze.count():,}")
print(f"Columnas: {len(df_bronze.columns)}")
df_bronze.printSchema()

In [0]:
from pyspark.sql.functions import col, year, month, round as spark_round

# ── Limpieza Silver ──────────────────────────────────────────────
df_silver = df_bronze \
    .filter((col("price") >= 10000) & (col("price") <= 10000000)) \
    .filter((year(col("date_of_transfer")) >= 2005) & 
            (year(col("date_of_transfer")) <= 2025)) \
    .withColumn("year",  year(col("date_of_transfer"))) \
    .withColumn("month", month(col("date_of_transfer"))) \
    .dropDuplicates(["transaction_id"])

print(f"Filas Bronze: {df_bronze.count():,}")
print(f"Filas Silver: {df_silver.count():,}")
print(f"Filas eliminadas: {df_bronze.count() - df_silver.count():,}")

In [0]:
# ── Guardar tabla Delta Silver ───────────────────────────────────
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.london_silver")

print("Tabla Silver guardada correctamente")